In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Qual é a receita total (em R$) somada de todos os filmes da base?
---

In [0]:
df_q1 = (
    spark.table("workspace.gold.fact_movies_performance")
    .agg(
        F.sum("receita_brl").alias("receita_total_brl")
    )
)

display(df_q1)

# 2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.
---

In [0]:
df_q2 = (
    spark.table("workspace.gold.fact_movies_performance")
    .join(
        spark.table("workspace.gold.dim_movies")
        .select("sk_movie_id", "titulo"),
        on="sk_movie_id",
        how="inner"
    )
    .select(
        "titulo",
        "popularidade"
    )
    .orderBy(
        F.col("popularidade").desc()
    )
    .limit(5)
)

display(df_q2)

# 3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.
---

In [0]:
df_q3 = (
    spark.table("workspace.gold.bridge_movie_genre")
    .join(
        spark.table("workspace.gold.dim_genres"),
        on="sk_genre_id",
        how="inner"
    )
    .groupBy("nome_genero")
    .agg(
        F.countDistinct("sk_movie_id").alias("qtd_filmes")
    )
    .orderBy(
        F.col("qtd_filmes").desc()
    )
)

display(df_q3)

# 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK()).
---

In [0]:
janela_ranking = Window.orderBy(
    F.col("receita_usd").desc()
)

df_q4 = (
    spark.table("workspace.gold.fact_movies_performance")
    .join(
        spark.table("workspace.gold.dim_movies")
        .select("sk_movie_id", "titulo"),
        on="sk_movie_id",
        how="inner"
    )
    .filter(
        F.col("receita_usd").isNotNull()
    )
    .withColumn(
        "ranking",
        F.rank().over(janela_ranking)
    )
    .select(
        "ranking",
        "titulo",
        "receita_usd",
        "receita_brl"
    )
    .filter(
        F.col("ranking") <= 10
    )
    .orderBy("ranking")
)

display(df_q4)

# 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?
---

In [0]:
df_movies = spark.table("workspace.gold.dim_movies")

# Encontrar a data mais recente entre os filmes lançados
data_maxima = (
    df_movies
    .filter(
        (F.col("status_filme") == "Lançado") &
        F.col("data_lancamento").isNotNull() &
        (F.col("data_lancamento") <= F.current_date())
    )
    .agg(
        F.max("data_lancamento").alias("data_max")
    )
    .first()["data_max"]
)

df_q5 = (
    spark.table("workspace.gold.bridge_movie_person")
    .join(
        spark.table("workspace.gold.dim_people"),
        on="sk_person_id",
        how="inner"
    )
    .join(
        df_movies.select(
            "sk_movie_id",
            "data_lancamento",
            "status_filme"
        ),
        on="sk_movie_id",
        how="inner"
    )
    .filter(
        (F.col("tipo_pessoa") == "Ator") &
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_maxima), -24)) &
        (F.col("data_lancamento") <= F.lit(data_maxima))
    )
    .groupBy("nome_pessoa")
    .agg(
        F.countDistinct("sk_movie_id").alias("qtd_participacoes")
    )
    .orderBy(
        F.col("qtd_participacoes").desc()
    )
    .limit(1)
)

display(df_q5)

# 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos?
---

In [0]:
df_q6 = (
    spark.table("workspace.gold.bridge_movie_company")
    .join(
        spark.table("workspace.gold.dim_companies"),
        on="sk_company_id",
        how="inner"
    )
    .join(
        df_movies.select(
            "sk_movie_id",
            "data_lancamento",
            "status_filme"
        ),
        on="sk_movie_id",
        how="inner"
    )
    .join(
        spark.table("workspace.gold.fact_movies_performance")
        .select(
            "sk_movie_id",
            "lucro_usd"
        ),
        on="sk_movie_id",
        how="inner"
    )
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_maxima), -60)) &
        (F.col("data_lancamento") <= F.lit(data_maxima)) &
        F.col("lucro_usd").isNotNull()
    )
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").alias("lucro_total_usd")
    )
    .orderBy(
        F.col("lucro_total_usd").desc()
    )
    .limit(1)
)

display(df_q6)

O lucro total dos filmes associados à produtora Universal, porque um mesmo filme pode ter várias produtoras na bridge.